# LeetCode 264 — Ugly Number II bằng Python

Link đề: https://leetcode.com/problems/ugly-number-ii/ · Độ khó: **Medium** · Chủ đề: Hash Table, Math, Dynamic Programming, Heap (Priority Queue)

Skeleton LeetCode:

```python
class Solution:
    def nthUglyNumber(self, n: int) -> int:
        ...
```

Ghi chú: đề, ví dụ và constraints lấy từ đề gốc; mọi đáp án trong notebook đều được kiểm chứng bằng code và đối chiếu với một lời giải brute force độc lập (duyệt từng số nguyên, chia hết cho 2, 3, 5 để kiểm tra "xấu số").

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: **số xấu (ugly number)** là số nguyên dương mà các ước nguyên tố chỉ thuộc `{2, 3, 5}` (số `1` cũng là số xấu); tìm số xấu **thứ `n`** theo thứ tự tăng dần.
2. Biết cách **kiểm tra một số có xấu không** bằng cách chia hết liên tiếp cho `2`, `3`, `5`, và thấy vì sao **duyệt từng số nguyên** không dùng được khi `n = 1690` (đáp án lên tới hơn 2 tỷ).
3. Nhận ra **cấu trúc sinh số**: mọi số xấu (trừ `1`) bằng một số xấu **nhỏ hơn** nhân với `2`, `3` hoặc `5`. Từ đó đi từ **duyệt + kiểm tra (baseline)** → **min-heap + tập `set` chống trùng** → **quy hoạch động ba con trỏ `O(n)`** → **sinh mọi số dạng `2^a·3^b·5^c` rồi sắp xếp** → **min-heap sinh duy nhất không cần `set`**.
4. Hiểu **ba con trỏ**: ba dãy `2·U`, `3·U`, `5·U` đều đã sắp, ta **trộn** chúng (như bước merge của merge sort) và loại trùng.
5. Tránh các lỗi kinh điển: kiểm tra xấu số sai (chỉ cần chia hết cho một trong 2, 3, 5), quên loại trùng (`6 = 2·3 = 3·2`), lệch một đơn vị chỉ số (`dp[n]` thay vì `dp[n-1]`), dùng `elif` ở bước dịch con trỏ.

## 2. Đề bài & ví dụ

Một **số xấu (ugly number)** là một số nguyên dương mà các thừa số nguyên tố của nó **chỉ** gồm `2`, `3` và `5`.

Cho số nguyên `n`, hãy trả về **số xấu thứ `n`** (đếm từ `1`, dãy sắp tăng dần).

**Ví dụ 1**

- Input: `n = 10`
- Output: `12`
- Giải thích: dãy 10 số xấu đầu tiên là `[1, 2, 3, 4, 5, 6, 8, 9, 10, 12]`.

**Ví dụ 2**

- Input: `n = 1`
- Output: `1`
- Giải thích: `1` không có thừa số nguyên tố nào, nên mọi thừa số nguyên tố của nó (không có cái nào) đều thuộc `{2, 3, 5}`; vì vậy `1` là số xấu.

**Ràng buộc:** `1 <= n <= 1690`.

In [1]:
from typing import List
import random
import heapq
from heapq import heappush, heappop

examples = [
    (10, 12),
    (1, 1),
]
for n, expected in examples:
    print(f"n={n}, kỳ vọng {expected}")

n=10, kỳ vọng 12
n=1, kỳ vọng 1


## 3. Kiến thức nền

### 3.1. Thừa số nguyên tố và kiểm tra "xấu số" bằng cách chia

Một số là xấu nếu sau khi **chia hết liên tục** cho `2`, rồi cho `3`, rồi cho `5` ta còn lại đúng `1`. Nếu còn lại số khác `1` thì nó có thừa số nguyên tố khác (`7`, `11`, ...). Ví dụ `12 = 2·2·3 → 1` (xấu), `14 = 2·7 → 7` (không xấu).

In [2]:
def is_ugly(x: int) -> bool:
    if x <= 0:
        return False
    for p in (2, 3, 5):
        while x % p == 0:        # chia hết nhiều lần cho p
            x //= p
    return x == 1                # còn lại 1 nghĩa là chỉ có thừa số 2, 3, 5


print([x for x in range(1, 31) if is_ugly(x)])
print(is_ugly(12), is_ugly(14), is_ugly(1), is_ugly(0))

[1, 2, 3, 4, 5, 6, 8, 9, 10, 12, 15, 16, 18, 20, 24, 25, 27, 30]
True False True False


### 3.2. Duyệt từng số nguyên để tìm số xấu thứ `n`: vì sao không đủ nhanh

Cách trực tiếp: đếm các số xấu khi tăng `x = 1, 2, 3, ...` cho tới khi đủ `n`. Với `n` nhỏ thì tốt, nhưng số xấu thứ `1690` là `2123366400` (hơn hai tỷ): ta phải kiểm tra hơn hai tỷ số, quá chậm. Số xấu **thưa dần**: chỉ chỉ khoảng `1690` số trong hơn 2 tỷ số.

In [3]:
def nth_by_scan(n):
    count, x = 0, 0
    while count < n:
        x += 1
        if is_ugly(x):
            count += 1
    return x


print([nth_by_scan(k) for k in range(1, 16)])
print("số xấu thứ 100:", nth_by_scan(100), "(phải kiểm tra", nth_by_scan(100), "số nguyên)")

[1, 2, 3, 4, 5, 6, 8, 9, 10, 12, 15, 16, 18, 20, 24]
số xấu thứ 100: 1536 (phải kiểm tra 1536 số nguyên)


### 3.3. Cấu trúc sinh: mỗi số xấu (khác 1) là một số xấu nhỏ hơn nhân `2`, `3` hoặc `5`

Nếu `x > 1` là số xấu thì `x` chia hết cho ít nhất một trong `2`, `3`, `5`; chia bớt thừa số đó ta được một số xấu **nhỏ hơn** `x`. Ngược lại nhân một số xấu với `2`, `3` hoặc `5` luôn cho số xấu. Vậy tập các số xấu là `{1}` cùng mọi số suy ra từ nhân liên tiếp: ta có thể **sinh** chúng theo thứ tự tăng thay vì kiểm tra từng số.

In [4]:
ugly = [x for x in range(1, 200) if is_ugly(x)]
for x in ugly[1:12]:
    parents = [x // p for p in (2, 3, 5) if x % p == 0 and is_ugly(x // p)]
    print(f"{x} = số xấu nhỏ hơn {parents} nhân thừa số tương ứng")

2 = số xấu nhỏ hơn [1] nhân thừa số tương ứng
3 = số xấu nhỏ hơn [1] nhân thừa số tương ứng
4 = số xấu nhỏ hơn [2] nhân thừa số tương ứng
5 = số xấu nhỏ hơn [1] nhân thừa số tương ứng
6 = số xấu nhỏ hơn [3, 2] nhân thừa số tương ứng
8 = số xấu nhỏ hơn [4] nhân thừa số tương ứng
9 = số xấu nhỏ hơn [3] nhân thừa số tương ứng
10 = số xấu nhỏ hơn [5, 2] nhân thừa số tương ứng
12 = số xấu nhỏ hơn [6, 4] nhân thừa số tương ứng
15 = số xấu nhỏ hơn [5, 3] nhân thừa số tương ứng
16 = số xấu nhỏ hơn [8] nhân thừa số tương ứng


### 3.4. Hàng đợi ưu tiên `heapq` (min-heap)

`heapq` quản lý một list như **min-heap**: `heappush(h, x)` thêm `x` trong `O(log n)`, `heappop(h)` lấy ra phần tử **nhỏ nhất** trong `O(log n)`, `h[0]` là phần tử nhỏ nhất (không lấy ra). Hữu ích khi cần luôn lấy giá trị nhỏ nhất trong một tập đang thay đổi.

In [5]:
h = []
for x in [5, 2, 9, 1, 7]:
    heappush(h, x)
print("nhỏ nhất:", h[0])
print([heappop(h) for _ in range(len(h))])    # [1, 2, 5, 7, 9]: lấy ra tăng dần

nhỏ nhất: 1
[1, 2, 5, 7, 9]


### 3.5. `set` để loại trùng và phép lặp nhiều con trỏ

`set` kiểm tra "đã có chưa" trong `O(1)` trung bình. Cần dùng khi một số có thể sinh ra bằng nhiều cách (`6 = 2·3 = 3·2`). Một kỹ thuật khác là dùng **nhiều con trỏ** `p2`, `p3`, `p5` chỉ vào các vị trí trong cùng một danh sách đã sinh; ta sẽ làm ở cách 3.

In [6]:
seen = set()
for a in (2, 3, 5):
    for b in (2, 3, 5):
        seen.add(a * b)
print(sorted(seen))          # [4, 6, 9, 10, 15, 25]: 6, 10, 15 chỉ xuất hiện một lần dù có hai cách sinh

[4, 6, 9, 10, 15, 25]


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Viết các số xấu theo thứ tự rồi để ý mỗi số đến từ đâu:

| thứ tự | số xấu | sinh từ |
|---|---|---|
| 1 | 1 | (gốc) |
| 2 | 2 | `1 × 2` |
| 3 | 3 | `1 × 3` |
| 4 | 4 | `2 × 2` |
| 5 | 5 | `1 × 5` |
| 6 | 6 | `2 × 3` hoặc `3 × 2` (trùng!) |
| 7 | 8 | `4 × 2` |
| 8 | 9 | `3 × 3` |
| 9 | 10 | `2 × 5` hoặc `5 × 2` (trùng!) |
| 10 | 12 | `4 × 3` hoặc `6 × 2` (trùng!) |

Quan sát: số xấu tiếp theo luôn là **số nhỏ nhất trong các tích `x·2`, `x·3`, `x·5` chưa dùng** (với `x` chạy qua các số xấu đã có). Cần cơ chế để (a) luôn lấy số nhỏ nhất (min-heap hoặc ba con trỏ) và (b) không đếm trùng.

In [7]:
ugly_list = [1]
candidates = {2, 3, 5}
for _ in range(9):
    nxt = min(candidates)
    ugly_list.append(nxt)
    candidates.remove(nxt)
    candidates.update({nxt * 2, nxt * 3, nxt * 5})
print(ugly_list)           # 10 số đầu tiên: [1, 2, 3, 4, 5, 6, 8, 9, 10, 12]

[1, 2, 3, 4, 5, 6, 8, 9, 10, 12]


## 5. Cách 1 — Brute Force / baseline: duyệt từng số nguyên và kiểm tra

**Ý tưởng.** Tăng `x` từ `1`, đếm các số xấu (`is_ugly`) cho tới khi đếm đủ `n`, trả `x`. Rõ ràng và chắc chắn đúng, nhưng chậm khi `n` lớn.

In [8]:
def nth_ugly_bruteforce(n: int) -> int:
    count = 0
    x = 0
    while count < n:
        x += 1
        if is_ugly(x):            # chia hết liên tiếp cho 2, 3, 5
            count += 1
    return x


for n, expected in examples:
    print(f"n={n} -> {nth_ugly_bruteforce(n)} (kỳ vọng {expected})")
print("n=200 ->", nth_ugly_bruteforce(200))

n=10 -> 12 (kỳ vọng 12)
n=1 -> 1 (kỳ vọng 1)
n=200 -> 16200


**Độ phức tạp (Cách 1).** Gọi `U(n)` là số xấu thứ `n`. Phải kiểm tra `U(n)` số nguyên, mỗi lần `O(log U)` phép chia: **Time `O(U(n) · log U(n))`**, **Space `O(1)`**. Với `n = 1690`, `U(n) ≈ 2.1·10⁹`: **không khả thi**. Ta chỉ dùng nó cho `n` nhỏ làm chuẩn đối chiếu.

**Nút thắt.** Ta kiểm tra rất nhiều số **không** xấu. Hãy **sinh trực tiếp** các số xấu theo thứ tự tăng.

## 6. Ý tưởng thuật toán / bước đột phá

**Sinh các số xấu theo thứ tự tăng dần.** Gọi `U` là dãy số xấu đã sắp. Từ mục 3.3: mọi số xấu `> 1` có dạng `u·2`, `u·3` hoặc `u·5` với `u ∈ U` đứng **trước** nó. Vậy `U` là **trộn của ba dãy đã sắp**: `2U`, `3U`, `5U` (kèm số `1` ở đầu), và **loại các phần tử trùng** (ví dụ `6` có trong cả `2U` và `3U`).

Hai cách trộn:

- **Min-heap:** đẩy các số sinh ra vào heap, lấy nhỏ nhất ra làm số xấu kế tiếp; dùng `set` để không đẩy trùng (Cách 2).
- **Ba con trỏ:** giữ `p2`, `p3`, `p5` là vị trí trong `U` của phần tử tiếp theo cần nhân với `2`, `3`, `5`. Số xấu kế tiếp là `min(U[p2]·2, U[p3]·3, U[p5]·5)`; **mọi** con trỏ có tích bằng giá trị nhỏ nhất đó đều dịch sang phải (đó chính là bước loại trùng). Chỉ `O(n)` (Cách 3).

In [9]:
n = 10
U = [1]
p2 = p3 = p5 = 0
for i in range(1, n):
    a, b, c = U[p2] * 2, U[p3] * 3, U[p5] * 5
    nxt = min(a, b, c)
    U.append(nxt)
    print(f"i={i}: {U[p2]}*2={a}, {U[p3]}*3={b}, {U[p5]}*5={c} -> chọn {nxt}")
    if nxt == a:
        p2 += 1
    if nxt == b:
        p3 += 1
    if nxt == c:
        p5 += 1
print(U)

i=1: 1*2=2, 1*3=3, 1*5=5 -> chọn 2
i=2: 2*2=4, 1*3=3, 1*5=5 -> chọn 3
i=3: 2*2=4, 2*3=6, 1*5=5 -> chọn 4
i=4: 3*2=6, 2*3=6, 1*5=5 -> chọn 5
i=5: 3*2=6, 2*3=6, 2*5=10 -> chọn 6
i=6: 4*2=8, 3*3=9, 2*5=10 -> chọn 8
i=7: 5*2=10, 3*3=9, 2*5=10 -> chọn 9
i=8: 5*2=10, 4*3=12, 2*5=10 -> chọn 10
i=9: 6*2=12, 4*3=12, 3*5=15 -> chọn 12
[1, 2, 3, 4, 5, 6, 8, 9, 10, 12]


## 7. Các cách tiếp theo

### Cách 2 — Min-heap + `set` chống trùng (cách của lời giải tham khảo)

**Giải thích.** Heap khởi đầu `[1]`, `seen = {1}`. Lặp `n` lần: `x = heappop(h)`; với mỗi `v ∈ {2, 3, 5}`, nếu `x·v` chưa trong `seen` thì thêm vào `seen` và `heappush`. Giá trị `x` bị lấy ra ở lần lặp thứ `n` là đáp án.

In [10]:
def nth_ugly_heap_set(n: int) -> int:
    h = [1]
    seen = {1}
    x = 1
    for _ in range(n):
        x = heappop(h)                          # số xấu nhỏ nhất chưa lấy
        for v in (2, 3, 5):
            nxt = x * v
            if nxt not in seen:                 # tránh đẩy trùng (6 = 2*3 = 3*2)
                seen.add(nxt)
                heappush(h, nxt)
    return x


for n, expected in examples:
    print(f"n={n} -> {nth_ugly_heap_set(n)} (kỳ vọng {expected})")

n=10 -> 12 (kỳ vọng 12)
n=1 -> 1 (kỳ vọng 1)


In [11]:
# Code nộp LeetCode cho Cách 2
from heapq import heappop, heappush


class Solution:
    def nthUglyNumber(self, n: int) -> int:
        h = [1]
        vis = {1}
        ans = 1
        for _ in range(n):
            ans = heappop(h)
            for v in [2, 3, 5]:
                nxt = ans * v
                if nxt not in vis:
                    vis.add(nxt)
                    heappush(h, nxt)
        return ans

**Độ phức tạp Cách 2:** mỗi lần lặp một `heappop` và tối đa ba `heappush`, mỗi thao tác `O(log n)` (heap có `O(n)` phần tử): **Time `O(n log n)`**, **Space `O(n)`** cho heap và `set`.

### Cách 3 — Quy hoạch động với ba con trỏ

**Giải thích.** Như mục 6. `dp[0] = 1`; với `i` từ `1` đến `n − 1`: `a, b, c = dp[p2]·2, dp[p3]·3, dp[p5]·5`; `dp[i] = min(a, b, c)`; rồi **ba lệnh `if` độc lập** (không `elif`): nếu `dp[i] == a` thì `p2 += 1`; nếu `== b` thì `p3 += 1`; nếu `== c` thì `p5 += 1`. Đáp án `dp[n − 1]`.

In [12]:
def nth_ugly_dp(n: int) -> int:
    dp = [1] * n                                  # dp[i] là số xấu thứ i + 1
    p2 = p3 = p5 = 0
    for i in range(1, n):
        a, b, c = dp[p2] * 2, dp[p3] * 3, dp[p5] * 5
        dp[i] = min(a, b, c)
        if dp[i] == a:                            # mọi con trỏ cho giá trị nhỏ nhất đều dịch
            p2 += 1
        if dp[i] == b:
            p3 += 1
        if dp[i] == c:
            p5 += 1
    return dp[n - 1]


for n, expected in examples:
    print(f"n={n} -> {nth_ugly_dp(n)} (kỳ vọng {expected})")

n=10 -> 12 (kỳ vọng 12)
n=1 -> 1 (kỳ vọng 1)


**Độ phức tạp Cách 3:** **Time `O(n)`**, **Space `O(n)`** cho mảng `dp`. Nhanh nhất; không cần `set`, không cần heap.

### Cách 4 — Sinh mọi số dạng `2^a · 3^b · 5^c` rồi sắp xếp

**Giải thích.** Số xấu chính là các số `2^a·3^b·5^c` (`a, b, c ≥ 0`). Vì đáp án lớn nhất `< 2³¹`, ta sinh mọi số dạng đó không vượt `2³¹ − 1` bằng ba vòng lặp lồng nhau (có `1691` số như vậy, trong đó `1690` số đầu là các đáp án cần thiết), sắp xếp, rồi tra bảng. Tính trước một lần, mỗi truy vấn `O(1)`.

In [13]:
def build_ugly_table(limit=2**31 - 1):
    table = []
    p2 = 1
    while p2 <= limit:
        p3 = p2
        while p3 <= limit:
            p5 = p3
            while p5 <= limit:
                table.append(p5)
                p5 *= 5
            p3 *= 3
        p2 *= 2
    table.sort()
    return table


UGLY_TABLE = build_ugly_table()
print("số phần tử:", len(UGLY_TABLE), "| 10 số đầu:", UGLY_TABLE[:10], "| số xấu thứ 1690:", UGLY_TABLE[1689])


def nth_ugly_table(n: int) -> int:
    return UGLY_TABLE[n - 1]


for n, expected in examples:
    print(f"n={n} -> {nth_ugly_table(n)} (kỳ vọng {expected})")

số phần tử: 1691 | 10 số đầu: [1, 2, 3, 4, 5, 6, 8, 9, 10, 12] | số xấu thứ 1690: 2123366400
n=10 -> 12 (kỳ vọng 12)
n=1 -> 1 (kỳ vọng 1)


**Độ phức tạp Cách 4:** tiền xử lý `O(M)` với `M` ≈ 1691 số sinh ra cộng sắp xếp `O(M log M)`; mỗi truy vấn **`O(1)`**; **Space `O(M)`**. Ưu điểm: rất đơn giản; nhược điểm: dựa vào việc biết trước giới hạn (`n ≤ 1690`, kết quả `< 2³¹`).

### Cách 5 — Min-heap sinh duy nhất, không cần `set`

**Giải thích.** Cách sinh để mỗi số chỉ được đẩy **đúng một lần**: từ `x` vừa lấy ra, luôn đẩy `2x`; đẩy `3x` chỉ khi `x` **không chia hết cho 2**; đẩy `5x` chỉ khi `x` không chia hết cho `2` và `3`. Lý do: mỗi số xấu `y = 2^a·3^b·5^c` có **một cha duy nhất**: bỏ một thừa số `2` nếu `a > 0` (cha là `y/2`), nếu không thì bỏ một thừa số `3` nếu `b > 0` (cha là `y/3`), nếu không thì bỏ một thừa số `5` (cha là `y/5`). Quy tắc trên chính là điều kiện để `x` là cha của `x·p`.

In [14]:
def nth_ugly_heap_unique(n: int) -> int:
    h = [1]
    x = 1
    for _ in range(n):
        x = heappop(h)
        heappush(h, x * 2)                      # 2x luôn có cha là x
        if x % 2 != 0:
            heappush(h, x * 3)                  # 3x có cha x khi x không chứa thừa số 2
            if x % 3 != 0:
                heappush(h, x * 5)              # 5x có cha x khi x không chứa thừa số 2, 3
    return x


for n, expected in examples:
    print(f"n={n} -> {nth_ugly_heap_unique(n)} (kỳ vọng {expected})")

n=10 -> 12 (kỳ vọng 12)
n=1 -> 1 (kỳ vọng 1)


**Độ phức tạp Cách 5:** **Time `O(n log n)`**, **Space `O(n)`**; không cần `set` nhưng đẩy ít hơn.

> Cách 3 là cách nộp bài tốt nhất (`O(n)`). Cách 2 là cách phổ biến, dễ nghĩ. Cách 4 phù hợp khi nhiều truy vấn. Cách 1 là chuẩn đối chiếu.

## 8. TRACE từng bước

Ta theo dõi Cách 3 (ba con trỏ) cho `n = 10`, rồi heap của Cách 2 cho `n = 7`.

In [15]:
def trace_dp(n):
    dp = [1] * n
    p2 = p3 = p5 = 0
    print(f"n={n}; dp[0]=1")
    for i in range(1, n):
        a, b, c = dp[p2] * 2, dp[p3] * 3, dp[p5] * 5
        dp[i] = min(a, b, c)
        moved = []
        if dp[i] == a:
            p2 += 1
            moved.append("p2")
        if dp[i] == b:
            p3 += 1
            moved.append("p3")
        if dp[i] == c:
            p5 += 1
            moved.append("p5")
        print(f"  i={i}: a={a:>2}, b={b:>2}, c={c:>2} -> dp[{i}]={dp[i]:>2}; dịch {'+'.join(moved)} => (p2,p3,p5)=({p2},{p3},{p5})")
    return dp[n - 1]


assert trace_dp(10) == 12

n=10; dp[0]=1
  i=1: a= 2, b= 3, c= 5 -> dp[1]= 2; dịch p2 => (p2,p3,p5)=(1,0,0)
  i=2: a= 4, b= 3, c= 5 -> dp[2]= 3; dịch p3 => (p2,p3,p5)=(1,1,0)
  i=3: a= 4, b= 6, c= 5 -> dp[3]= 4; dịch p2 => (p2,p3,p5)=(2,1,0)
  i=4: a= 6, b= 6, c= 5 -> dp[4]= 5; dịch p5 => (p2,p3,p5)=(2,1,1)
  i=5: a= 6, b= 6, c=10 -> dp[5]= 6; dịch p2+p3 => (p2,p3,p5)=(3,2,1)
  i=6: a= 8, b= 9, c=10 -> dp[6]= 8; dịch p2 => (p2,p3,p5)=(4,2,1)
  i=7: a=10, b= 9, c=10 -> dp[7]= 9; dịch p3 => (p2,p3,p5)=(4,3,1)
  i=8: a=10, b=12, c=10 -> dp[8]=10; dịch p2+p5 => (p2,p3,p5)=(5,3,2)
  i=9: a=12, b=12, c=15 -> dp[9]=12; dịch p2+p3 => (p2,p3,p5)=(6,4,2)


In [16]:
def trace_heap(n):
    h, seen = [1], {1}
    print(f"n={n}; heap ban đầu {h}")
    for step in range(1, n + 1):
        x = heappop(h)
        pushed = []
        for v in (2, 3, 5):
            nxt = x * v
            if nxt not in seen:
                seen.add(nxt)
                heappush(h, nxt)
                pushed.append(nxt)
        print(f"  lần {step}: lấy {x}, đẩy {pushed}, heap (không sắp) = {sorted(h)}")
    return x


assert trace_heap(7) == 8

n=7; heap ban đầu [1]
  lần 1: lấy 1, đẩy [2, 3, 5], heap (không sắp) = [2, 3, 5]
  lần 2: lấy 2, đẩy [4, 6, 10], heap (không sắp) = [3, 4, 5, 6, 10]
  lần 3: lấy 3, đẩy [9, 15], heap (không sắp) = [4, 5, 6, 9, 10, 15]
  lần 4: lấy 4, đẩy [8, 12, 20], heap (không sắp) = [5, 6, 8, 9, 10, 12, 15, 20]
  lần 5: lấy 5, đẩy [25], heap (không sắp) = [6, 8, 9, 10, 12, 15, 20, 25]
  lần 6: lấy 6, đẩy [18, 30], heap (không sắp) = [8, 9, 10, 12, 15, 18, 20, 25, 30]
  lần 7: lấy 8, đẩy [16, 24, 40], heap (không sắp) = [9, 10, 12, 15, 16, 18, 20, 24, 25, 30, 40]


Bảng tóm tắt (Cách 3, `n = 10`):

| `i` | `dp[p2]·2` | `dp[p3]·3` | `dp[p5]·5` | `dp[i]` | con trỏ dịch |
|---|---|---|---|---|---|
| 1 | 2 | 3 | 5 | 2 | `p2` |
| 2 | 4 | 3 | 5 | 3 | `p3` |
| 3 | 4 | 6 | 5 | 4 | `p2` |
| 4 | 6 | 6 | 5 | 5 | `p5` |
| 5 | 6 | 6 | 10 | 6 | `p2` **và** `p3` (loại trùng) |
| 6 | 8 | 9 | 10 | 8 | `p2` |
| 7 | 10 | 9 | 10 | 9 | `p3` |
| 8 | 10 | 12 | 10 | 10 | `p2` **và** `p5` (loại trùng) |
| 9 | 12 | 12 | 15 | 12 | `p2` **và** `p3` |

Ở `i = 5` giá trị `6` đến từ cả `2·3` lẫn `3·2` nên **cả hai con trỏ** cùng dịch: đó là cơ chế loại trùng.

## 9. Code nộp LeetCode

Bản nộp dùng Cách 3 (ba con trỏ, `O(n)`). Đúng class `Solution`, đúng method `nthUglyNumber(self, n: int) -> int`.

In [17]:
class Solution:
    def nthUglyNumber(self, n: int) -> int:
        dp = [1] * n                              # dp[i] = số xấu thứ i + 1
        p2 = p3 = p5 = 0                          # vị trí cần nhân với 2, 3, 5 tiếp theo
        for i in range(1, n):
            a, b, c = dp[p2] * 2, dp[p3] * 3, dp[p5] * 5
            dp[i] = min(a, b, c)
            if dp[i] == a:                        # ba lệnh if độc lập: loại trùng
                p2 += 1
            if dp[i] == b:
                p3 += 1
            if dp[i] == c:
                p5 += 1
        return dp[n - 1]


for n, expected in examples:
    assert Solution().nthUglyNumber(n) == expected
print("PASS ví dụ đề cho class Solution; số xấu thứ 1690 =", Solution().nthUglyNumber(1690))

PASS ví dụ đề cho class Solution; số xấu thứ 1690 = 2123366400


## 10. Vì sao thuật toán đúng

- **Cấu trúc:** gọi `U` là dãy số xấu sắp tăng. Mọi `u ∈ U` với `u > 1` có thể viết `u = v·p` với `v ∈ U`, `v < u`, `p ∈ {2, 3, 5}` (chia bớt một thừa số). Ngược lại `v·p ∈ U` với mọi `v ∈ U`. Vậy `U = {1} ∪ 2U ∪ 3U ∪ 5U`.
- **Ba con trỏ là bước merge:** `2U`, `3U`, `5U` đều là các dãy **tăng** (nhân dãy tăng với hằng số dương). `p2` trỏ tới phần tử nhỏ nhất của `2U` chưa được đưa vào `dp` (tức `U[p2]·2` nhỏ nhất chưa dùng); tương tự `p3`, `p5`. Số xấu kế tiếp là phần tử nhỏ nhất chưa dùng của hợp ba dãy, tức `min(a, b, c)`. Phần tử này chắc chắn `>` số vừa thêm vì ta luôn dịch mọi con trỏ có giá trị bằng nó (không còn bản sao).
- **Loại trùng:** nếu `dp[i] == a == b` (như `6`) thì cả `p2` và `p3` dịch, nên `dp[p2]·2` và `dp[p3]·3` không lặp lại giá trị `6` ở lần sau. Dùng `elif` chỉ dịch một con trỏ nên `6` xuất hiện lần nữa (lỗi).
- **Đủ và không bỏ sót:** mỗi số xấu `u` có một cha nhỏ hơn `v`; khi `v` được thêm vào `dp`, con trỏ tương ứng sẽ có lúc trỏ tới `v` và sinh `u` trước khi vượt qua nó; vì ta luôn lấy nhỏ nhất, `u` được thêm đúng thứ tự.
- **Heap + set (Cách 2):** heap luôn chứa mọi số xấu đã sinh nhưng chưa lấy; số nhỏ nhất trong heap là số xấu kế tiếp (mọi số nhỏ hơn đã được sinh vì cha của nó nhỏ hơn và đã bị lấy). `set` đảm bảo mỗi số chỉ vào heap một lần.
- **Cách 5:** mỗi số xấu `y` có cha duy nhất theo quy tắc ở mục Cách 5, nên được đẩy đúng một lần mà không cần `set`.

## 11. Phân tích Time / Space Complexity

Gọi `n` là chỉ số cần tìm (`≤ 1690`), `U(n)` là số xấu thứ `n`.

- **Cách 1 (duyệt + kiểm tra):** Time `O(U(n) · log U(n))`, Space `O(1)`; không dùng cho `n` lớn.
- **Cách 2 (min-heap + `set`):** Time `O(n log n)`, Space `O(n)`.
- **Cách 3 (ba con trỏ):** Time `O(n)`, Space `O(n)`.
- **Cách 4 (sinh `2^a 3^b 5^c` + sắp xếp):** tiền xử lý `O(M log M)` (với `M ≈ 1690`), truy vấn `O(1)`, Space `O(M)`.
- **Cách 5 (heap sinh duy nhất):** Time `O(n log n)`, Space `O(n)`.
- Với `n ≤ 1690` mọi cách ngoại trừ Cách 1 đều chạy tức thì. Có `1691` số xấu không vượt `2³¹ − 1`; số thứ `1690` là `2123366400` và số thứ `1691` là `2125764000`.

## 12. Bộ test

Ta gom các cách vào từ điển `impls`. `oracle` là cách quét từng số nguyên với `is_ugly` cho `n ≤ 400`, và với `n` lớn hơn so với một **danh sách số xấu độc lập** dựng bằng sàng: đánh dấu mọi số `≤ B` mà chỉ có thừa số 2, 3, 5 (quét `x` từ `1` tới `B` bằng `is_ugly`) với `B = 3·10⁶`.

In [18]:
sol = Solution()
impls = {
    "heap_set": nth_ugly_heap_set,
    "dp": nth_ugly_dp,
    "table": nth_ugly_table,
    "heap_unique": nth_ugly_heap_unique,
    "submit": sol.nthUglyNumber,
}

B = 3 * 10**6
SCAN_LIST = [x for x in range(1, B + 1) if is_ugly(x)]          # danh sách số xấu <= 3*10^6 (độc lập)
print("số số xấu <= 3*10^6 (quét từng số):", len(SCAN_LIST))


def oracle(n):
    if n <= len(SCAN_LIST):
        return SCAN_LIST[n - 1]
    raise ValueError("n vượt phạm vi quét")


for n, expected in examples:
    assert oracle(n) == expected
    for name, fn in impls.items():
        assert fn(n) == expected, (name, n, fn(n), expected)
assert nth_ugly_bruteforce(10) == 12
print("PASS ví dụ của đề (mọi cách và oracle độc lập)")

số số xấu <= 3*10^6 (quét từng số): 622
PASS ví dụ của đề (mọi cách và oracle độc lập)


### 12.1. Các trường hợp biên

In [19]:
def check_all(n, expected, label):
    for name, fn in impls.items():
        assert fn(n) == expected, (label, name, fn(n), expected)
    print("PASS", label, "->", expected)


check_all(1, 1, "n = 1 nhỏ nhất")
check_all(2, 2, "n = 2")
check_all(7, 8, "n = 7: số 7 không phải số xấu nên nhảy từ 6 lên 8")
check_all(10, 12, "n = 10 (đề)")
check_all(11, 15, "n = 11")
check_all(1690, 2123366400, "n = 1690 lớn nhất của đề: 2123366400")
for n in (1500, 1600, 1689):
    assert len({fn(n) for fn in impls.values()}) == 1, n
print("PASS các n lớn (1500, 1600, 1689): mọi cách cho cùng kết quả")
assert all(UGLY_TABLE[i] < UGLY_TABLE[i + 1] for i in range(len(UGLY_TABLE) - 1)) and len(UGLY_TABLE) == 1691

PASS n = 1 nhỏ nhất -> 1
PASS n = 2 -> 2
PASS n = 7: số 7 không phải số xấu nên nhảy từ 6 lên 8 -> 8
PASS n = 10 (đề) -> 12
PASS n = 11 -> 15
PASS n = 1690 lớn nhất của đề: 2123366400 -> 2123366400
PASS các n lớn (1500, 1600, 1689): mọi cách cho cùng kết quả


### 12.2. Tính chất và phản ví dụ

In [20]:
random.seed(264)

# Tính chất 1: dãy tăng chặt; mọi phần tử là số xấu; số xấu thứ n >= n
seq = [sol.nthUglyNumber(k) for k in range(1, 1691)]
assert all(seq[i] < seq[i + 1] for i in range(len(seq) - 1))
assert all(is_ugly(x) for x in seq) and all(seq[i] >= i + 1 for i in range(len(seq)))
print("PASS tính chất: dãy tăng chặt, mọi phần tử đều là số xấu, U(n) >= n")

# Tính chất 2: nếu x là số xấu thì 2x, 3x, 5x cũng là số xấu và có mặt trong dãy
S = set(seq)
for x in seq[:200]:
    for v in (2, 3, 5):
        if x * v <= seq[-1]:
            assert x * v in S
print("PASS tính chất: đóng với phép nhân 2, 3, 5")

# Tính chất 3: giữa hai số xấu liên tiếp không có số xấu nào (so với quét độc lập)
for k in range(len(SCAN_LIST) - 1):
    assert seq[k] == SCAN_LIST[k]
    if SCAN_LIST[k] > 10**6:
        break
print("PASS tính chất: khớp danh sách quét độc lập")

# Phản ví dụ: kiểm tra xấu số sai (chỉ chia hết cho 2, 3 hoặc 5)
def sai_is_ugly(x):
    return x > 0 and (x % 2 == 0 or x % 3 == 0 or x % 5 == 0)

print("số 14 -> sai:", sai_is_ugly(14), "| đúng:", is_ugly(14))
assert sai_is_ugly(14) and not is_ugly(14)

PASS tính chất: dãy tăng chặt, mọi phần tử đều là số xấu, U(n) >= n
PASS tính chất: đóng với phép nhân 2, 3, 5
PASS tính chất: khớp danh sách quét độc lập
số 14 -> sai: True | đúng: False


### 12.3. Đối chiếu ngẫu nhiên (có seed) với oracle độc lập

In [21]:
random.seed(2026)
ns = list(range(1, 401)) + [random.randint(1, 1690) for _ in range(300)] + [random.randint(1, len(SCAN_LIST)) for _ in range(5)]
for n in ns:
    if n > len(SCAN_LIST):
        continue
    expected = oracle(n)
    for name, fn in impls.items():
        assert fn(n) == expected, (name, n, fn(n), expected)
for n in range(1, 200):                       # baseline brute force thật sự (quét từng số) cho n nhỏ
    assert nth_ugly_bruteforce(n) == nth_ugly_dp(n)
print("PASS", len(ns), "giá trị n: mọi cách khớp oracle quét độc lập; brute force khớp n < 200")

PASS 705 giá trị n: mọi cách khớp oracle quét độc lập; brute force khớp n < 200


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Kiểm tra xấu số bằng "chia hết cho 2, 3 hoặc 5"

In [22]:
def sai_kiem_tra(n):
    count, x = 0, 0
    while count < n:
        x += 1
        if x % 2 == 0 or x % 3 == 0 or x % 5 == 0 or x == 1:      # sai: chỉ cần một ước 2, 3 hoặc 5
            count += 1
    return x


print("n=10 -> sai:", sai_kiem_tra(10), "| đúng:", Solution().nthUglyNumber(10))

n=10 -> sai: 12 | đúng: 12


**Vì sao sai:** số xấu cần **mọi** thừa số nguyên tố thuộc `{2, 3, 5}`, không chỉ có một ước thuộc tập đó. `14 = 2·7` chia hết cho `2` nhưng có thừa số `7` nên không xấu. Phải chia hết liên tiếp rồi kiểm tra phần còn lại bằng `1`.

### Lỗi 2: Dùng `elif` khi dịch con trỏ (không loại trùng)

In [23]:
def sai_elif(n):
    dp = [1] * n
    p2 = p3 = p5 = 0
    for i in range(1, n):
        a, b, c = dp[p2] * 2, dp[p3] * 3, dp[p5] * 5
        dp[i] = min(a, b, c)
        if dp[i] == a:
            p2 += 1
        elif dp[i] == b:                         # sai: bỏ qua việc dịch p3 khi a == b
            p3 += 1
        elif dp[i] == c:
            p5 += 1
    return dp


print("12 số đầu (sai):", sai_elif(12))
print("12 số đầu (đúng):", [Solution().nthUglyNumber(k) for k in range(1, 13)])

12 số đầu (sai): [1, 2, 3, 4, 5, 6, 6, 8, 9, 10, 10, 12]
12 số đầu (đúng): [1, 2, 3, 4, 5, 6, 8, 9, 10, 12, 15, 16]


**Vì sao sai:** khi hai tích bằng nhau (`6 = 2·3 = 3·2`) chỉ một con trỏ dịch, tích còn lại vẫn là `6` ở vòng sau, nên `6` bị **lặp lại**. Dãy sai: `1, 2, 3, 4, 5, 6, 6, 8, ...`. Mỗi con trỏ có giá trị bằng nhỏ nhất đều phải dịch (ba lệnh `if` độc lập).

### Lỗi 3: Lệch một đơn vị chỉ số (`dp[n]` thay vì `dp[n - 1]`)

In [24]:
def sai_lech_mot(n):
    dp = [1] * (n + 1)
    p2 = p3 = p5 = 0
    for i in range(1, n + 1):
        a, b, c = dp[p2] * 2, dp[p3] * 3, dp[p5] * 5
        dp[i] = min(a, b, c)
        if dp[i] == a:
            p2 += 1
        if dp[i] == b:
            p3 += 1
        if dp[i] == c:
            p5 += 1
    return dp[n]                                  # sai: dp[n] là số xấu thứ n + 1 (dp[0] = 1 là số thứ nhất)


print("n=10 -> sai:", sai_lech_mot(10), "| đúng:", Solution().nthUglyNumber(10))

n=10 -> sai: 15 | đúng: 12


**Vì sao sai:** `dp[0] = 1` là số xấu **thứ nhất**, nên số xấu thứ `n` là `dp[n − 1]`. Dùng `dp[n]` cho số thứ `n + 1` (ở đây `15` thay vì `12`).

### Lỗi 4: Dùng heap mà không loại trùng

In [25]:
def sai_heap_khong_set(n):
    h = [1]
    x = 1
    for _ in range(n):
        x = heappop(h)
        for v in (2, 3, 5):
            heappush(h, x * v)                   # sai: 6 = 2*3 và 3*2 bị đẩy hai lần
    return x


print("n=10 -> sai:", sai_heap_khong_set(10), "| đúng:", Solution().nthUglyNumber(10))

n=10 -> sai: 10 | đúng: 12


**Vì sao sai:** cùng một số có nhiều cách sinh, nên xuất hiện nhiều lần trong heap và bị lấy ra **nhiều lần** như nhiều số xấu khác nhau, làm đếm sai. Cần `set` (Cách 2) hoặc quy tắc sinh duy nhất (Cách 5).

## 14. Mẹo làm nhanh & nhận dạng pattern

- "Số thứ `n` có thừa số nguyên tố giới hạn" (ugly, humble, super ugly, magical numbers) → **trộn các dãy tăng `p·U`** bằng nhiều con trỏ hoặc heap.
- Dãy được **sinh từ chính nó** (`U` chứa `2U`, `3U`, `5U`) → DP một mảng với con trỏ cho mỗi "nguồn".
- Muốn loại trùng khi trộn: dịch **mọi** con trỏ có tích bằng giá trị nhỏ nhất vừa chọn (dùng nhiều `if`, không phải `elif`).
- Khi cần lấy "nhỏ nhất chưa dùng" từ tập thay đổi: **min-heap**; tránh trùng bằng `set` hoặc quy tắc cha duy nhất.
- Giới hạn nhỏ cố định (`n ≤ 1690`) cho phép **tính trước bảng** rồi tra cứu.
- Không bao giờ duyệt từng số khi đáp án lớn hơn rất nhiều `n` (đáp án ~ `2·10⁹`).

## 15. Biến thể của bài

1. **Ugly Number I (LeetCode 263):** kiểm tra một số có phải số xấu không.
2. **Super Ugly Number (LeetCode 313):** các thừa số nguyên tố thuộc một danh sách `primes` cho trước.
3. **Đếm số xấu `≤ N`** (không tìm số thứ `n`).
4. **Humble numbers** (thừa số `2, 3, 5, 7`).
5. **Số xấu nằm trong đoạn `[lo, hi]`** (dựa vào bảng đã sinh).

In [26]:
def super_ugly(n, primes):
    """Biến thể 2/4: số thứ n mà mọi thừa số nguyên tố nằm trong primes (nhiều con trỏ)."""
    dp = [1] * n
    ptr = [0] * len(primes)
    for i in range(1, n):
        cands = [dp[ptr[k]] * primes[k] for k in range(len(primes))]
        dp[i] = min(cands)
        for k in range(len(primes)):
            if cands[k] == dp[i]:
                ptr[k] += 1
    return dp[n - 1]


def count_ugly_upto(N):
    """Biến thể 3: số xấu <= N (dùng bảng đã sinh bằng nhân)."""
    return sum(1 for v in UGLY_TABLE if v <= N)


def ugly_in_range(lo, hi):
    """Biến thể 5."""
    return [v for v in UGLY_TABLE if lo <= v <= hi]


def is_ugly_generic(x, primes):
    if x <= 0:
        return False
    for p in primes:
        while x % p == 0:
            x //= p
    return x == 1


assert super_ugly(10, [2, 3, 5]) == 12
assert super_ugly(12, [2, 7, 13, 19]) == 32                    # ví dụ của LeetCode 313
assert count_ugly_upto(30) == len([x for x in range(1, 31) if is_ugly(x)]) == 18
assert ugly_in_range(10, 30) == [10, 12, 15, 16, 18, 20, 24, 25, 27, 30]
random.seed(15)
for _ in range(100):
    primes = sorted(random.sample([2, 3, 5, 7, 11], random.randint(3, 4)))   # >= 3 thừa số để dãy không thưa quá
    n = random.randint(1, 40)
    count, x = 0, 0
    while count < n:
        x += 1
        if is_ugly_generic(x, primes):
            count += 1
    assert super_ugly(n, primes) == x, (primes, n)
print("PASS biến thể: super ugly, đếm <= N, đoạn [lo, hi]")

PASS biến thể: super ugly, đếm <= N, đoạn [lo, hi]


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Duyệt + kiểm tra từng số | `is_ugly` rồi đếm | `O(U(n) log U)` | `O(1)` | oracle, `n` nhỏ |
| 2. Min-heap + `set` | lấy nhỏ nhất, đẩy `×2, ×3, ×5` chưa gặp | `O(n log n)` | `O(n)` | dễ nghĩ, phổ biến |
| 3. DP ba con trỏ | trộn `2U`, `3U`, `5U`, dịch mọi con trỏ bằng nhỏ nhất | **`O(n)`** | `O(n)` | **cách nộp bài** |
| 4. Sinh `2^a 3^b 5^c` + sắp xếp | bảng tính trước 1690 số | tiền xử lý `O(M log M)`, truy vấn `O(1)` | `O(M)` | nhiều truy vấn, giới hạn cố định |
| 5. Heap sinh duy nhất | chỉ đẩy `3x` nếu `x` lẻ, `5x` nếu không chia hết 2 và 3 | `O(n log n)` | `O(n)` | heap không cần `set` |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: `is_ugly(30)`, `is_ugly(49)`, `Solution().nthUglyNumber(6)`, `Solution().nthUglyNumber(9)`, `[x for x in range(1, 11) if is_ugly(x)]`.

**Bài 2 (tính tay).** Với ba con trỏ, tính tay `dp[0..8]` (9 số xấu đầu tiên), ghi lại `(a, b, c)`, giá trị chọn và con trỏ nào dịch ở mỗi bước.

**Bài 3 (điền TODO).** Hoàn thành `ugly_todo(n)` theo Cách 3 (ba con trỏ).

In [27]:
def ugly_todo(n):
    dp = [1] * n
    p2 = p3 = p5 = 0
    for i in range(1, n):
        # TODO 1: a, b, c = dp[p2] * 2, dp[p3] * 3, dp[p5] * 5
        # TODO 2: dp[i] = min(a, b, c)
        # TODO 3: ba lệnh if độc lập: nếu dp[i] == a thì p2 += 1; nếu == b thì p3 += 1; nếu == c thì p5 += 1
        pass
    return dp[n - 1]

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi (các TODO chưa điền nên kết quả chưa đúng).

**Bài 4 (tự viết lại).** Viết `ugly_heap2(n)` theo Cách 2 (min-heap + `set`) mà không nhìn code mẫu. Kiểm tra bằng hai ví dụ của đề.

**Bài 5 (biến thể / mini challenge).** (a) Viết `nth_humble(n)` cho các thừa số `2, 3, 5, 7` (dùng ba con trỏ mở rộng thành bốn con trỏ). (b) Viết `ugly_between(lo, hi)` trả về số lượng số xấu trong `[lo, hi]` mà không quét từng số nguyên.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: số xấu chỉ có thừa số 2, 3, 5; `49 = 7·7`. Bài 2: mỗi bước lấy nhỏ nhất của ba tích.
- **Mức 2 (vừa):** Bài 1: `True`, `False`, `6`, `10`, `[1, 2, 3, 4, 5, 6, 8, 9, 10]`. Bài 2: xem bảng ở mục 8 (chín số đầu `1, 2, 3, 4, 5, 6, 8, 9, 10`). Bài 3: ba TODO trong comment.
- **Mức 3 (sâu):** Bài 4: heap `[1]`, `seen = {1}`; lặp `n` lần `heappop` rồi đẩy `x*2, x*3, x*5` nếu chưa trong `seen`. Bài 5(a): `super_ugly(n, [2, 3, 5, 7])` (bốn con trỏ). Bài 5(b): dùng bảng `UGLY_TABLE` và `bisect` để đếm.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [28]:
from bisect import bisect_left, bisect_right

# Bài 1
assert is_ugly(30) is True and is_ugly(49) is False
assert Solution().nthUglyNumber(6) == 6 and Solution().nthUglyNumber(9) == 10
assert [x for x in range(1, 11) if is_ugly(x)] == [1, 2, 3, 4, 5, 6, 8, 9, 10]

# Bài 2
assert [Solution().nthUglyNumber(k) for k in range(1, 10)] == [1, 2, 3, 4, 5, 6, 8, 9, 10]

# Bài 3
def ugly_todo_ans(n):
    dp = [1] * n
    p2 = p3 = p5 = 0
    for i in range(1, n):
        a, b, c = dp[p2] * 2, dp[p3] * 3, dp[p5] * 5
        dp[i] = min(a, b, c)
        if dp[i] == a:
            p2 += 1
        if dp[i] == b:
            p3 += 1
        if dp[i] == c:
            p5 += 1
    return dp[n - 1]

for n in range(1, 300):
    assert ugly_todo_ans(n) == nth_ugly_dp(n)

# Bài 4
def ugly_heap2(n):
    h, seen = [1], {1}
    x = 1
    for _ in range(n):
        x = heappop(h)
        for v in (2, 3, 5):
            if x * v not in seen:
                seen.add(x * v)
                heappush(h, x * v)
    return x

for n, expected in examples:
    assert ugly_heap2(n) == expected
assert all(ugly_heap2(n) == nth_ugly_dp(n) for n in range(1, 200))

# Bài 5(a), 5(b)
def nth_humble(n):
    return super_ugly(n, [2, 3, 5, 7])

def ugly_between(lo, hi):
    return bisect_right(UGLY_TABLE, hi) - bisect_left(UGLY_TABLE, lo)

assert [nth_humble(k) for k in range(1, 11)] == [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
assert nth_humble(11) == 12 and nth_humble(13) == 15
for lo, hi in [(1, 10), (10, 30), (100, 1000), (1, 2**31 - 1)]:
    assert ugly_between(lo, hi) == sum(1 for v in UGLY_TABLE if lo <= v <= hi)
print("PASS đáp án phần 19")

PASS đáp án phần 19


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (trộn `2U`, `3U`, `5U` bằng ba con trỏ)

```python
class Solution:
    def nthUglyNumber(self, n: int) -> int:
        dp = [1] * n
        p2 = p3 = p5 = 0
        for i in range(1, n):
            a, b, c = dp[p2] * 2, dp[p3] * 3, dp[p5] * 5
            dp[i] = min(a, b, c)
            if dp[i] == a:
                p2 += 1
            if dp[i] == b:
                p3 += 1
            if dp[i] == c:
                p5 += 1
        return dp[n - 1]
```

**Checklist**

- [ ] `dp[0] = 1`; đáp án là `dp[n − 1]` (không phải `dp[n]`)?
- [ ] Ba lệnh `if` **độc lập** để dịch mọi con trỏ có giá trị bằng nhỏ nhất (loại trùng), không dùng `elif`?
- [ ] Nếu dùng heap: có `set` (hoặc quy tắc cha duy nhất) để không đẩy trùng?
- [ ] Không duyệt từng số nguyên (đáp án lớn nhất `2123366400`)?
- [ ] Hiểu "số xấu" = mọi thừa số nguyên tố thuộc `{2, 3, 5}` (không phải chỉ chia hết cho một trong chúng) và `1` là số xấu?
- [ ] Đã thử: `n = 1`, `n = 7` (nhảy qua `7`), `n = 1690`?
- [ ] Đã đối chiếu với brute force quét từng số (cho `n` nhỏ) và danh sách quét độc lập?
- [ ] Skeleton đúng `class Solution` / `def nthUglyNumber(self, n: int) -> int`?